# 01 — Loading the Friends corpus and computing VAD scores

First of three notebooks walking through the pipeline.  Here we load the two data sources, project each utterance into the NRC Valence–Arousal–Dominance space, and build per-character SEP trajectories that the CSD detector consumes downstream.

**Reading order:**  `01_data_and_vad` → `02_csd_detection_and_validation` → `03_case_study_scene9`

## Setup

The model engine lives in `../code/model/`.  We add it to `sys.path` so the project modules can be imported directly.

In [ ]:
import os, sys
NB_DIR = os.path.abspath('.')
sys.path.insert(0, os.path.join(NB_DIR, '..', 'code', 'model'))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Loading the Emory NLP *Friends* corpus

The corpus contains 97 episodes (Seasons 1–4) annotated at the utterance level with seven categories: *Joyful, Peaceful, Powerful, Neutral, Scared, Sad, Mad*.  After removing 90 group-tagged (`#ALL#`) and 71 multi-speaker utterances, the working corpus contains 12,445 utterances.  Analysis focuses on the six main characters (~9,776 utterances).

The categorical labels are reserved for **post-hoc validation only** — the detector never sees them.

In [ ]:
from data_loader import load_and_sort, filter_speakers
from config import JSON_PATHS, MAIN_CHARACTERS

all_utts = filter_speakers(load_and_sort(JSON_PATHS))
print(f'Total utterances after filtering: {len(all_utts):,}')
print(f'Main characters: {", ".join(MAIN_CHARACTERS)}')

## 2. The NRC VAD lexicon

NRC VAD v2.1 maps ~55,000 English entries to continuous (Valence, Arousal, Dominance) scores in [−1, +1].  We use a **multi-word-expression-first greedy match**: try bigram → fall back to unigram → skip if unmatched.  The utterance-level VAD is the mean across matched entries.

In [ ]:
from vad_engine import load_vad_lexicon, utterance_vad

vad_lex, mwe_lex = load_vad_lexicon('../data/NRC-VAD-Lexicon-v2.1.txt')
print(f'Unigrams: {len(vad_lex):,}    Bigrams (MWEs): {len(mwe_lex):,}')

### Sanity check — a few utterances from Scene 9 of S02E18

These three lines preview the central problem of the project: the lexicon's signed-affect reading does not match the conversational role of the speaker.

In [ ]:
samples = [
    ('Joey, suffering',     [['I', 'don', 't', 'feel', 'like', 'talkin']]),
    ('Rachel, comforting',  [['Oh', 'c', 'mon', 'Joey', 'we', 'care', 'about', 'you']]),
    ('Phoebe, ironic',      [['sorry', 'about', 'your', 'death']]),
]
print(f"{'Utterance':<22} {'V':>8} {'A':>8} {'D':>8} {'SEP':>8}")
for name, tokens in samples:
    vad, cov = utterance_vad(tokens, vad_lex, mwe_lex)
    if vad:
        v, a, d = vad
        sep = v + 0.5*d
        print(f'{name:<22} {v:>+8.3f} {a:>+8.3f} {d:>+8.3f} {sep:>+8.3f}')

Notice already: Joey's *Sad* line scores **positive** SEP because the lexicon reads `feel` and `like` as positive — while Phoebe's *Peaceful* (comforting) line scores **negative** because `sorry` and `death` carry strong negative valence.  This is the lexical substrate of the structural error documented in Notebook 2.

## 3. Building per-character trajectories

`build_trajectories` aligns each character's utterances chronologically (within episode → within season) and applies same-episode same-speaker mean imputation for utterances with zero lexicon coverage.  This is the input the CSD detector expects.

In [ ]:
from vad_engine import build_trajectories

trajs = build_trajectories(all_utts, vad_lex, mwe_lex, MAIN_CHARACTERS)
for char in MAIN_CHARACTERS:
    rows = trajs[char]
    seps = [r['vad'][0] + 0.5*r['vad'][2] for r in rows]
    print(f'{char:<18} n={len(rows):>4}    '
          f'mu_SEP={np.mean(seps):+.3f}    sigma_SEP={np.std(seps):.3f}')

## 4. Per-character SEP distributions

These histograms are the empirical density that the next notebook's potential landscape `U(SEP) = -log p(SEP)` will be built on.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=True, sharey=True)
COLORS = {
    'Phoebe Buffay':  '#9b59b6', 'Chandler Bing':  '#2980b9',
    'Joey Tribbiani': '#e67e22', 'Ross Geller':    '#16a085',
    'Rachel Green':   '#c0392b', 'Monica Geller':  '#d35400',
}
for ax, char in zip(axes.flatten(), MAIN_CHARACTERS):
    seps = [r['vad'][0] + 0.5*r['vad'][2] for r in trajs[char]]
    ax.hist(seps, bins=40, color=COLORS[char], alpha=0.85)
    ax.axvline(np.mean(seps), color='black', linestyle='--', linewidth=0.8)
    ax.set_title(f'{char}  (mu={np.mean(seps):+.3f})', fontsize=10)
    ax.grid(alpha=0.3)
fig.suptitle('Per-character SEP distributions  -  the input to the CSD detector',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

All six characters share a basin near SEP ≈ +0.15 — the mildly positive default of a sitcom register.  Differences across characters are subtle on the raw histogram and become more interpretable in the log-density potential view in Notebook 2.

## 5. Final database export

The full per-utterance VAD trajectory is written to `final_database/friends_main_chars_vad_trajectory.csv` (11 columns × 9,776 rows).  Running the script below regenerates the file from scratch — it is also the source of the small scene-9 subset used in Notebook 3.

In [ ]:
%run ../code/export_final_database.py

---

**Next:** `02_csd_detection_and_validation.ipynb` runs the two-layer CSD detector on these trajectories, manually validates all 20 candidate tipping segments against a literature-grounded codebook, and reproduces Figures 1–3 of the report.